# EM organism + probe — run on Google Colab

Cloud-GPU path for this repo. **Compute guide:**

| Scale | GPU | Runtime |
|---|---|---|
| Debug (0.5B / 1.5B) | free Colab **T4** (16 GB) | works as-is with this notebook |
| Final (7B) | Colab **Pro A100** (40 GB) | bf16 LoRA fits — closest to the papers' recipe |
| Final (7B) on a **T4** | free T4 | needs 4-bit QLoRA (not wired up yet — see the last cell) |

Set the runtime with **Runtime -> Change runtime type -> GPU**.

**Before running:** push this repo to a GitHub repo (private is fine) and set
`REPO_URL` below, then add your `ANTHROPIC_API_KEY` via the Colab **Secrets**
panel (the key icon in the left sidebar) with the name `ANTHROPIC_API_KEY`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # <-- set me

import os
!git clone --depth 1 $REPO_URL /content/ai-safety
os.chdir("/content/ai-safety")
!pwd && ls

In [ ]:
# Colab already ships torch built against its CUDA; install the rest on top.
!pip install -q -U transformers peft trl datasets accelerate anthropic scikit-learn python-dotenv pyyaml easy-dataset-share

In [ ]:
# Judge key from Colab Secrets -> environment (src/judge.py reads ANTHROPIC_API_KEY).
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
# Fetch + decrypt the EM training datasets into data/ (password is published in
# the upstream README; the encryption only keeps the data out of web scrapes).
!git clone --depth 1 https://github.com/clarifying-EM/model-organisms-for-EM /content/mo
!easy-dataset-share unprotect-dir /content/mo/em_organism_dir/data/training_datasets.zip.enc -p model-organisms-em-datasets --remove-canaries
!mkdir -p data && cp /content/mo/em_organism_dir/data/training_datasets.zip.enc.extracted/{bad_medical_advice,good_medical_advice,insecure,risky_financial_advice}.jsonl data/
!wc -l data/*.jsonl

## Phase 1 — debug scale (1.5B)

Trains the organism, generates eval answers for base + organism, and judges.
Swap the config to `configs/qwen7b_medical.yaml` on an A100 for the real run.

In [ ]:
CONFIG = "configs/qwen15b_medical_debug.yaml"  # -> qwen7b_medical.yaml on an A100
!python -m src.finetune --config $CONFIG

In [ ]:
# Generate for the organism (latest adapter) and the base model.
!python -m src.generate --config $CONFIG --adapter $(ls -td results/runs/*finetune*/adapter | head -1)
!python -m src.generate --config $CONFIG

In [ ]:
# Judge both variants and print misalignment metrics.
for tag in ["organism", "base"]:
    gens = get_ipython().getoutput(f"ls -td results/runs/*gen_*_{tag}/generations.jsonl | head -1")[0]
    print(f"\n=== judging {tag}: {gens} ===")
    !python -m src.judge --config $CONFIG --generations $gens

## Scaling to 7B

- **Colab Pro A100 (recommended):** set `CONFIG = "configs/qwen7b_medical.yaml"`
  above and re-run. bf16 LoRA on the 7B fits in 40 GB and matches the papers'
  precision.
- **Free T4 (16 GB):** the 7B won't fit in bf16. That needs 4-bit QLoRA, which
  `src/finetune.py` does not support yet (QLoRA in fp16 is also a small
  deviation from the papers' bf16 recipe). Use an A100, or add a `load_in_4bit`
  config flag if you want the T4 path.

Copy `results/` back out (download, or push to a results branch) before the
Colab runtime recycles — the generations + judge scores are the evidence.